In [0]:
from pyspark.sql import functions as F

BASE = "abfss://silver@plantationsimulatorrg.dfs.core.windows.net"

EXPECTED = {
    "weather": 6483,
    "harvest": 9112,
    "fertilizer": 9000,
    "equipment": 10000,
    "hr": 2000,
    "finance": 12000,
}

print("=" * 70)
print("PHASE 3 — SILVER VERIFICATION")
print("=" * 70)

grand_total = 0
all_pass = True

for source, expected in EXPECTED.items():
    path = f"{BASE}/{source}"
    df = spark.read.format("delta").load(path)

    actual = df.count()
    duplicates = actual - df.dropDuplicates().count()
    has_ingested_at = "_ingested_at" in df.columns
    schema_ok = len(df.columns) > 0

    row_pass = actual == expected
    all_pass &= row_pass and duplicates == 0 and has_ingested_at and schema_ok
    grand_total += actual

    print(f"\n{'─' * 70}")
    print(f"DATASET: {source}")
    print(f"Path:    {path}")
    print(f"Rows:    {actual:,} / expected {expected:,} -> {'PASS' if row_pass else 'FAIL'}")
    print(f"Columns: {len(df.columns)} -> {'PASS' if schema_ok else 'FAIL'}")
    print(f"Duplicates: {duplicates:,} -> {'PASS' if duplicates == 0 else 'CHECK'}")
    print(f"_ingested_at: {'PASS' if has_ingested_at else 'FAIL'}")

    print("\nSchema:")
    df.printSchema()

    print("Sample rows:")
    display(df.limit(5))

print(f"\n{'=' * 70}")
print(f"TOTAL SILVER ROWS: {grand_total:,} / 48,595")
print(f"FINAL RESULT: {'PASS — ALL SILVER DATASETS VERIFIED' if all_pass and grand_total == 48595 else 'FAIL — REVIEW ABOVE'}")
print("=" * 70)